In [4]:
import os

# Get the current working directory
current_dir = os.getcwd()
print("Current working directory:", current_dir)

import pandas as pd
import scanpy as sc
import anndata as ad
from tqdm import tqdm
import matplotlib.pyplot as plt # import matplotlib to visualize our qc metrics
import subprocess
import sys
import seaborn as sns
import numpy as np
from scipy.sparse import csr_matrix
import scanpy.external as sce
from sklearn.metrics import silhouette_score
import datetime
from collections import defaultdict
import scipy.sparse as sp
import gffutils 
import gffutils 

# Load gene gtf file for gene length 
gtf_file = "/gpfs/commons/groups/knowles_lab/Karin/Leaflet-analysis-WD/TabulaSenis/genome_files/gencode.vM19/genes/genes.gtf"
db_file="/gpfs/commons/groups/knowles_lab/Karin/Leaflet-analysis-WD/TabulaSenis/genome_files/GENCODE_vM19"

today = datetime.datetime.now()
today = today.strftime("%Y-%m-%d")

Current working directory: /gpfs/commons/home/kisaev/Leaflet-analysis/Mouse_Splicing_Foundation/GeneExpression/10X_prep


In [5]:
# Read in the Mouse foundation dataset so we just keep the same genes as the ones there 
mouse_foundation_path = "/gpfs/commons/groups/knowles_lab/Karin/Leaflet-analysis-WD/MOUSE_SPLICING_FOUNDATION/MODEL_INPUT/052025/mouse_foundation_data_20250502_155802_ge.h5ad"
mouse_foundation = sc.read_h5ad(mouse_foundation_path)

# TMS 10X matrix
tms = "/gpfs/commons/projects/knowles_singlecell_splicing/TabulaSenis/data/AWS/processed_for_scanpy/tabulamurissenisdropletofficialrawobj.h5ad"
tms_adata = sc.read_h5ad(tms)
tms_adata.obs["dataset"] = "tabula_muris_senis"
tms_adata.obs["cell_id"] = tms_adata.obs.index.values
# Reset the index of adata.obs to integers and drop the old index
tms_adata.obs.reset_index(drop=True, inplace=True)
tms_adata.var["gene_symbol"] = tms_adata.var.index

# Filter to genes present in mouse foundation reference
tms_adata = tms_adata[:, tms_adata.var["gene_symbol"].isin(mouse_foundation.var["gene_symbol"])].copy()
tms_adata.layers["raw_counts"] = tms_adata.X.copy()

# Make sure to add library size to .obsm and .obs and have the same broad cell types
# Add column to gene expression AnnData for total library size using the raw_counts layer
library_size = np.asarray(tms_adata.layers["raw_counts"].sum(axis=1)).flatten()
tms_adata.obs["library_size"] = library_size # 1D column 
tms_adata.obsm["X_library_size"] = library_size[:, np.newaxis]
print(f"Done getting library sizes for all cells using length adjusted counts!")

/gpfs/commons/home/kisaev/miniconda3/envs/LeafletSC/lib/python3.10/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
/gpfs/commons/home/kisaev/miniconda3/envs/LeafletSC/lib/python3.10/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


Done getting library sizes for all cells using length adjusted counts!


In [6]:
# Save object under MouseFoundationProject
timestamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
output_dir="/gpfs/commons/groups/knowles_lab/Karin/Leaflet-analysis-WD/MOUSE_SPLICING_FOUNDATION/MODEL_INPUT/052025/10X"
print(f"Output directory: {output_dir}", flush=True)

base_filename = f"TMS_10X_data_{timestamp}"
ge_file = os.path.join(output_dir, f"{base_filename}_ge.h5ad")

print(f"Saving gene expression AnnData to: {ge_file}", flush=True)
tms_adata.write(ge_file, compression="lzf")

Output directory: /gpfs/commons/groups/knowles_lab/Karin/Leaflet-analysis-WD/MOUSE_SPLICING_FOUNDATION/MODEL_INPUT/052025/10X
Saving gene expression AnnData to: /gpfs/commons/groups/knowles_lab/Karin/Leaflet-analysis-WD/MOUSE_SPLICING_FOUNDATION/MODEL_INPUT/052025/10X/TMS_10X_data_20250511_085538_ge.h5ad
